# Lab 4 · Ridge, and the coefficients you cannot trust

**What you'll build:** the normal equations, a demonstration that their answers
wobble on this data, and the one-line fix that appears in
`src/pipeline/explain.py` as `1e-6 * trace(cov)`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labgrader import panel, grade_lab, TARGET, CLIMATE

df = panel()
print(df.shape, "rows x columns")
print("target column:", TARGET)

## 1. The climate features are almost the same variable

Hot summers are dry summers. Dry summers have high evaporative demand. High
demand means a big climatic moisture deficit. These are not independent
measurements of the world — they are four views of one thing.

In [ ]:
C = df[CLIMATE].corr()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(C, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(CLIMATE))); ax.set_xticklabels(CLIMATE, rotation=90, fontsize=8)
ax.set_yticks(range(len(CLIMATE))); ax.set_yticklabels(CLIMATE, fontsize=8)
fig.colorbar(im, ax=ax, shrink=.8)
ax.set_title("Correlation between climate predictors")
plt.tight_layout(); plt.show()

off = C.where(~np.eye(len(C), dtype=bool)).abs()
print("strongest pairs:")
print(off.unstack().sort_values(ascending=False).head(6).round(3))

## 2. What that does to the arithmetic

Fitting a linear model means solving

$$(X^\top X)\, w = X^\top y$$

If two columns of $X$ are nearly identical, $X^\top X$ is nearly **singular** —
hard to invert stably. The measure of "how close to impossible" is the
**condition number**. A condition number of $10^k$ means you can lose about $k$
of your ~16 digits of precision.

### ✏️ Assignment 1 — `COND_CLIMATE`

Set `COND_CLIMATE` to the condition number of the full climate feature matrix.
One call: `np.linalg.cond`.

In [ ]:
COND_CLIMATE = None   # >>> YOUR TURN

print(f"condition number: {COND_CLIMATE:.3e}")
print(f"roughly {np.log10(COND_CLIMATE):.0f} digits of precision at risk "
      f"(you have about 16)")

> Around $10^4$ — bad enough to matter, not bad enough for the arithmetic to
> visibly fall apart. So don't reach for the dramatic version of this story. The
> honest problem with collinearity here is not that the computer gets the wrong
> answer; it is that **the answer is unstable**, and an unstable coefficient
> cannot be interpreted.
>
> Here is how to see that. Resample the *rows* with replacement — a bootstrap —
> refit, and watch how much each coefficient moves. Do it with two predictors,
> then with two near-duplicates added.

In [ ]:
# The four predictors used for the rest of this notebook.
X = df[["PPT_sm", "Tmax_sm", "CMD_sm", "Eref_sm"]].to_numpy()
y = df[TARGET].to_numpy()

def coef_spread(cols, n=300, seed=3):
    X = df[cols].to_numpy()
    rng = np.random.default_rng(seed)
    fits = []
    for _ in range(n):
        i = rng.integers(0, len(X), len(X))
        fits.append(np.linalg.lstsq(np.c_[X[i], np.ones(len(i))], y[i], rcond=None)[0][:len(cols)])
    f = np.array(fits)
    return pd.DataFrame({"mean": f.mean(0), "sd": f.std(0),
                         "sd/|mean|": np.abs(f.std(0) / f.mean(0))}, index=cols)

print("Two predictors, cond = %.1e" % np.linalg.cond(df[["PPT_sm", "Tmax_sm"]].to_numpy()))
print(coef_spread(["PPT_sm", "Tmax_sm"]).round(6), "\n")

four = ["PPT_sm", "Tmax_sm", "CMD_sm", "logPPT_sm"]
print("Add two near-duplicates, cond = %.1e" % np.linalg.cond(df[four].to_numpy()))
print(coef_spread(four).round(6))

> **Read the `sd/|mean|` column.** With two predictors the coefficients are
> reasonably determined. Add `CMD_sm` and `logPPT_sm` — which correlate with
> `PPT_sm` at 0.99 — and the standard deviation of the `Tmax_sm` coefficient
> ends up **many times its own mean**. Its sign is not even reliably determined.
>
> That is what collinearity costs you. The model's *predictions* barely change;
> its *explanations* become worthless. If you ever say "precipitation matters
> more than temperature" from coefficients like these, someone should ask you
> this exact question.

## 3. The fix: add a little to the diagonal

Ridge regression solves

$$(X^\top X + \lambda I)\, w = X^\top y$$

That $\lambda I$ nudges the matrix away from singular. Two readings, both true:

- **Numerically:** it makes the matrix invertible again.
- **Statistically:** it penalises large coefficients, so the fit stops playing
  huge positives against huge negatives and settles on something modest.

You pay a little bias for a large drop in variance. On collinear data that is
almost always the right trade.

### ✏️ Assignment 2 — `ridge_solve`

Return `(w, intercept)`.

Do **not** put a column of ones in `X` — you would penalise the intercept, which
is wrong. Instead:

1. Centre: `Xc = X - X.mean(0)`, `yc = y - y.mean()`.
2. Solve `(Xc.T @ Xc + lam * I) w = Xc.T @ yc` with `np.linalg.solve`.
3. Recover `intercept = y.mean() - X.mean(0) @ w`.

In [ ]:
def ridge_solve(X, y, lam):
    """Ridge regression by the normal equations. Returns (coefficients, intercept)."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    # >>> YOUR TURN
    raise NotImplementedError

In [ ]:
# Check against scikit-learn.
from sklearn.linear_model import Ridge

for lam in (1e-6, 1.0, 100.0):
    w, b0 = ridge_solve(X, y, lam)
    sk = Ridge(alpha=lam).fit(X, y)
    print(f"lam={lam:<8g}  max coef gap {np.abs(w - sk.coef_).max():.2e}   "
          f"intercept gap {abs(b0 - sk.intercept_):.2e}")

In [ ]:
# The coefficient path: watch the penalty pull everything toward zero.
lams = np.logspace(-8, 4, 60)
paths = np.array([ridge_solve(X, y, l)[0] for l in lams])

fig, ax = plt.subplots(figsize=(7, 4.5))
for i, name in enumerate(["PPT_sm", "Tmax_sm", "CMD_sm", "Eref_sm"]):
    ax.plot(lams, paths[:, i], lw=2, label=name)
ax.set_xscale("log"); ax.axhline(0, color="k", lw=.5)
ax.set_xlabel("lambda"); ax.set_ylabel("coefficient"); ax.legend(fontsize=9)
ax.set_title("Ridge shrinks the fight between collinear predictors")
plt.show()

## 4. Where this shows up in the real code

`src/pipeline/explain.py` regularises a covariance matrix before inverting it,
and scales the ridge to the matrix rather than fixing it at a constant:

```python
cov + 1e-6 * np.trace(cov) / cov.shape[0] * np.eye(...)
```

Scaling matters. A fixed `lam=1e-6` means something completely different for a
matrix of millimetres of rain than for one of degrees. Tying it to the trace
makes it *relative* — a fixed fraction of the matrix's own scale.

---
## Grade it

In [ ]:
grade_lab(4, globals())

### What you should be able to say out loud

- The climate predictors are strongly collinear (pairs at r = 0.99), so
  $X^\top X$ is ill-conditioned — about $10^4$.
- The damage is **unstable coefficients**, not wrong predictions. Show it with a
  bootstrap, not with a condition number alone.
- Ridge adds $\lambda I$: numerically it restores invertibility, statistically it
  penalises large coefficients.
- The penalty must **not** apply to the intercept — centre instead.
- Scale $\lambda$ to the matrix (`trace`), never hard-code it.